TF-IDF scores each word in an error message by how useful it is for telling categories apart. A word that shows up in almost every error (like "error" or "line") gets a low score, since it doesn't help distinguish one category from another. A word that's rare but specific to certain errors (like "subscriptable" or "iterable") gets a high score, since seeing it strongly suggests a particular category. This works well here because our error messages are short and technical — a handful of distinctive words usually carry most of the signal.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
os.chdir(os.path.join(os.getcwd(), "..", ".."))  #  adjust working directory

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
print("TF-IDF import successful")

TF-IDF import successful


In [3]:
from client.runner import get_output_error, parse_error
from sanitizer.sanitizer import sanitize
import pandas as pd

scripts_and_labels = [
    ("ml_engine/data/raw/Syntax_Error.py", "syntax_error"),
    ("ml_engine/data/raw/Type_Error.py", "type_error"),
    ("ml_engine/data/raw/None_Type_Error.py", "none_type_error"),
    ("ml_engine/data/raw/Key_Error.py", "key_error"),
    ("ml_engine/data/raw/Index_Error.py", "index_error"),
    ("ml_engine/data/raw/Attribute_Error.py", "attribute_error"),
    ("ml_engine/data/raw/Module_Not_Found_Error.py", "module_not_found"),
    ("ml_engine/data/raw/File_Not_Found_Error.py", "file_not_found"),
    ("ml_engine/data/raw/permission_Error.py", "permission_error"),
    ("ml_engine/data/raw/value_Error.py", "value_error"),
    ("ml_engine/data/raw/network_Error.py", "network_error"),
    ("ml_engine/data/raw/Recursion_Error.py", "other_error"),
]

rows = []
for script_path, label in scripts_and_labels:
    error = get_output_error(script_path)
    result = parse_error(error)
    if result:
        error_type, message = result
        sanitized_message = sanitize(message)
        rows.append({"text": sanitized_message, "label": label})

df = pd.DataFrame(rows)
df.to_csv("ml_engine/data/labeled/dataset.csv", index=False)
print(df["label"].value_counts())

label
syntax_error        1
type_error          1
none_type_error     1
key_error           1
index_error         1
attribute_error     1
module_not_found    1
file_not_found      1
permission_error    1
value_error         1
network_error       1
other_error         1
Name: count, dtype: int64


In [4]:
print(df.head(12))

                                                 text             label
0                                        expected ':'      syntax_error
1   unsupported operand type(s) for +: 'int' and '...        type_error
2              'NoneType' object is not subscriptable   none_type_error
3                                                 'd'         key_error
4                             list index out of range       index_error
5            'BuyBook' object has no attribute 'loan'   attribute_error
6                       No module named 'prettytable'  module_not_found
7   [Errno 2] No such file or directory: 'non_exis...    file_not_found
8        [Errno 13] Permission denied: 'readonly.txt'  permission_error
9   invalid literal for int() with base 10: 'twent...       value_error
10  HTTPConnectionPool(host='127.0.0.1', port=1): ...     network_error
11                   maximum recursion depth exceeded       other_error
